In [ ]:
import pandas as pd
import numpy as np
import os

# ── Raw data directory ─────────────────────────────────────────────────────
RAW_DIR = "../data"

FILES = {
    "monday":       "Monday-WorkingHours.pcap_ISCX.csv",
    "bruteforce":   "Tuesday-WorkingHours.pcap_ISCX.csv",
    "dos":          "Wednesday-workingHours.pcap_ISCX.csv",
    "web_attacks": "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "infiltration":  "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "botnet":       "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "portscan":     "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "ddos":         "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
}
  
SOURCE_NAMES = {
    "monday":       "Monday",
    "bruteforce":   "BruteForce",
    "dos":          "DoS",
    "infiltration": "Infiltration",
    "web_attacks":  "WebAttacks",
    "botnet":       "Botnet",
    "portscan":     "PortScan",
    "ddos":         "DDoS",
}

print("Config loaded")

Config loaded


In [5]:
def preprocess_binary(df, source_name):
    """
    Clean a raw CICIDS2017 DataFrame and add binary label + source tag.
    
    Steps:
      1. Strip whitespace from column names
      2. Drop duplicate rows
      3. Drop rows with missing values
      4. Replace +/-inf with NaN, then drop again
      5. Add Label_Binary  (0 = BENIGN, 1 = attack)
      6. Add Source_File   (tracks which file each row came from)
    """
    df = df.copy()
    df.columns = df.columns.str.strip()
    df = df.drop_duplicates()
    df = df.dropna()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()
    df["Label_Binary"] = (df["Label"] != "BENIGN").astype(int)
    df["Source_File"]  = source_name
    return df

print("preprocess_binary() defined")

preprocess_binary() defined


In [6]:
def load_all_datasets(keys=None):
    """
    Load and preprocess CICIDS2017 files.
    Returns a dict mapping key -> cleaned DataFrame.
    Missing files are skipped with a warning.

    Args:
        keys: list of keys to load, or None to load all 8.

    Example:
        datasets = load_all_datasets()
        df_web      = datasets["web_attacks"]
        df_portscan = datasets["portscan"]

        # Load only what you need
        datasets = load_all_datasets(["web_attacks", "portscan", "ddos"])
    """
    if keys is None:
        keys = list(FILES.keys())

    print(f"Loading {len(keys)} dataset(s): {keys}\n")

    datasets = {}
    for key in keys:
        path = os.path.join(RAW_DIR, FILES[key])
        print(f"  [{key}]  {FILES[key]}")
        try:
            raw = pd.read_csv(path, low_memory=False)
            df  = preprocess_binary(raw, SOURCE_NAMES[key])
            datasets[key] = df
            attack_rate = df["Label_Binary"].mean() * 100
            print(f"           raw {raw.shape}  ->  clean {df.shape}  "
                  f"(attack rate {attack_rate:.1f}%)")
        except FileNotFoundError:
            print(f"           SKIPPED - not found in {RAW_DIR}")

    print(f"\n  {len(datasets)}/{len(keys)} datasets loaded.")
    return datasets

print("load_all_datasets() defined")

load_all_datasets() defined


## How to use in other notebooks

Add this at the top of any notebook:

```python
%run 00_data_loader.ipynb

# Load all 8 files
datasets = load_all_datasets()

# Or load only what you need
datasets = load_all_datasets(["web_attacks", "portscan", "ddos"])

df_web      = datasets["web_attacks"]
df_portscan = datasets["portscan"]
```